# Mamba Classifier Training NotebookTraining AML detector with Mamba + MLP on Elliptic2 dataset

In [ ]:
import sys, os, torch, numpy as np, pandas as pd
from pathlib import Path
import random, time
from tqdm import tqdm

# Find project root
ROOT_DIR = Path.cwd()
for p in [ROOT_DIR] + list(ROOT_DIR.parents):
    if (p / "src").exists() and (p / "data").exists():
        ROOT_DIR = p
        break
os.chdir(ROOT_DIR)
print(f"Working: {os.getcwd()}")

from src.dataset.elliptic_dataset import FastEllipticDataset
from src.models import create_model
from src.models.loss import get_loss_function
from src.utils.config import MODEL_CONFIG, TRAINING_CONFIG, LOSS_CONFIG
from src.utils.metrics import compute_metrics
from torch.utils.data import DataLoader

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## Configuration

In [ ]:
CONFIG = {
    # Data
    'graph_dir': 'data/processed/graph',
    'sequences_dir': 'data/processed/sequences',
    'index_dir': 'data/processed/index',
    
    # Training
    'device': 'cpu',
    'batch_size': 128,
    'num_epochs': 50,
    'learning_rate': 5e-4,
    'weight_decay': 1e-4,
    'num_workers': 0,
    
    # Loss
    'loss_type': 'focal',
    'class_weights': [1.0, 5000.0],
    'threshold': 0.15,
    'focal_gamma': 1.0,
    'focal_alpha': 0.75,
    
    # Early stopping
    'early_stopping_patience': 20,
    'val_interval': 1,
    
    'seed': 42,
}

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
print(f"Config: batch={CONFIG['batch_size']}, lr={CONFIG['learning_rate']}, epochs={CONFIG['num_epochs']}")

## Load Dataset

In [ ]:
train_dataset = FastEllipticDataset(
    graph_dir=CONFIG['graph_dir'],
    sequences_dir=CONFIG['sequences_dir'],
    index_dir=CONFIG['index_dir'],
    split='train'
)
val_dataset = FastEllipticDataset(
    graph_dir=CONFIG['graph_dir'],
    sequences_dir=CONFIG['sequences_dir'],
    index_dir=CONFIG['index_dir'],
    split='val'
)
test_dataset = FastEllipticDataset(
    graph_dir=CONFIG['graph_dir'],
    sequences_dir=CONFIG['sequences_dir'],
    index_dir=CONFIG['index_dir'],
    split='test'
)
print(f"Train: {len(train_dataset):,}, Val: {len(val_dataset):,}, Test: {len(test_dataset):,}")

## Create DataLoaders

In [ ]:
device = torch.device(CONFIG['device'])
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], collate_fn=FastEllipticDataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], collate_fn=FastEllipticDataset.collate_fn)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], collate_fn=FastEllipticDataset.collate_fn)
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}")

## Create Model

In [ ]:
model_config = {
    **MODEL_CONFIG,
}
model = create_model(model_config).to(device)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## Optimizer & Loss

In [ ]:
class_weights = torch.tensor(CONFIG['class_weights'], dtype=torch.float32, device=device)
criterion = get_loss_function(
    loss_type=CONFIG['loss_type'],
    class_weights=class_weights,
    focal_gamma=CONFIG.get('focal_gamma', 2.0),
    focal_alpha=CONFIG.get('focal_alpha', 0.25)
)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
print(f"Loss: {CONFIG['loss_type']}, weights={CONFIG['class_weights']}")

## Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, grad_clip=0.5):
    model.train()
    total_loss, num_batches = 0.0, 0
    for sequences, labels, _ in loader:
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, labels)
        if torch.isnan(loss): continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / max(num_batches, 1)

def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    all_preds, all_probs, all_labels, total_loss, num_batches = [], [], [], 0.0, 0
    with torch.no_grad():
        for sequences, labels, _ in loader:
            sequences, labels = sequences.to(device), labels.to(device)
            logits = model(sequences)
            loss = criterion(logits, labels)
            if not torch.isnan(loss): total_loss += loss.item(); num_batches += 1
            all_preds.append(logits.argmax(1).cpu())
            all_probs.append(torch.softmax(logits, 1)[:, 1].cpu())
            all_labels.append(labels.cpu())
    
    y_true, y_pred, y_prob = torch.cat(all_labels), torch.cat(all_preds), torch.cat(all_probs)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    
    # Threshold tuning
    pred_thresh = (y_prob > threshold).long()
    metrics_thresh = compute_metrics(y_true, pred_thresh, y_prob)
    if metrics_thresh['f1'] > metrics['f1']:
        metrics.update(metrics_thresh)
    
    metrics['loss'] = total_loss / max(num_batches, 1)
    return metrics

print("Functions defined")

## Training Loop

In [ ]:
from pathlib import Path
checkpoint_dir = Path("checkpoints"); checkpoint_dir.mkdir(exist_ok=True)

history = {'train_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_auc_pr': []}
best_f1, patience, start = 0.0, 0, time.time()

print("Starting training...")
for epoch in range(1, CONFIG['num_epochs'] + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    if epoch % CONFIG['val_interval'] == 0:
        val_metrics = evaluate(model, val_loader, criterion, device, CONFIG['threshold'])
        history['train_loss'].append(train_loss)
        history['val_f1'].append(val_metrics['f1'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_auc_pr'].append(val_metrics.get('auc_pr', 0.0))
        
        print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | f1={val_metrics['f1']:.4f} | prec={val_metrics['precision']:.4f} | rec={val_metrics['recall']:.4f} | auc_pr={val_metrics.get('auc_pr', 0):.4f} | {time.time()-t0:.0f}s")
        
        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            patience = 0
            torch.save({'epoch': epoch, 'model': model.state_dict(), 'metrics': val_metrics}, checkpoint_dir / 'best.pt')
            print(f"   -> Saved best (f1={best_f1:.4f})")
        else:
            patience += 1
        
        if patience >= CONFIG['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch}")
            break
    else:
        print(f"Epoch {epoch:02d} | loss={train_loss:.4f}")

print(f"\nDone! Best F1: {best_f1:.4f} in {(time.time()-start)/60:.1f} min")

## Test Evaluation

In [ ]:
if (checkpoint_dir / 'best.pt').exists():
    ckpt = torch.load(checkpoint_dir / 'best.pt', map_location=device)
    model.load_state_dict(ckpt['model'])
    test_metrics = evaluate(model, test_loader, criterion, device, CONFIG['threshold'])
    print(f"Test: f1={test_metrics['f1']:.4f} | prec={test_metrics['precision']:.4f} | rec={test_metrics['recall']:.4f} | auc_pr={test_metrics.get('auc_pr', 0):.4f}")
else:
    print("No checkpoint found")